# Cross-model analysis

Uses the runs of all four models (no API calls). Produces:

1. **Cross-model agreement**: pairwise Krippendorff's α between the models' baseline labels (part 3).
2. **Figure 3**: agreement with the human majority vote in the baseline condition, four models + ensemble.
3. **Figure 4**: LLM–human vs. human–human agreement, four models + ensemble.
4. **Table 2**: each alternative condition vs. the baseline, averaged over the four models (range in parentheses), plus one full table per model for the appendix.
5. **Sources of variation**: one summary table comparing run-to-run, prompt, few-shot and model differences.

Everything is written to `runs/cross_model/`. Needs `make_tables.py` and `make_figures.py` in this folder, and the ensemble files from `make_ensembles.py` (optional).

In [ ]:
%load_ext autoreload
%autoreload 2
import itertools, os
import numpy as np
import pandas as pd
import make_tables as mt
import make_figures as mf

# display name -> folder with that model's runs (edit the folders to match yours)
MODELS = {
    "gpt-6-luna":           "runs/gpt-6-luna",
    "Claude Haiku 5.5":     "runs/claude-haiku-5-5",
    "Gemma-4-31B-it":       "runs/gemma-4-31b-it-w4a16",
    "Qwen2.5-32B-Instruct": "runs/Qwen__Qwen2.5-32B-Instruct",
}
ENSEMBLE_TABLE1 = "runs/ensemble/table1_ensembles.csv"     # from make_ensembles.py
ENSEMBLE_NAME = "Ensemble (mean of 4 3-model ensembles)"
mt.HUMAN = ("https://raw.githubusercontent.com/supertrashpanda/llm-moral-rhetoric-tutorial/"
            "refs/heads/main/test_full.csv")
OUT = "runs/cross_model"
os.makedirs(OUT, exist_ok=True)
LABELS = mt.LABELS + ["foundations (mean)"]

## 1. Cross-model agreement (baseline condition)

α between every pair of models (6 pairs), per label, on the paragraphs all four models coded. The human–human column is the mean over the three coder pairs, for comparison.

In [ ]:
def baseline(folder):
    mt.OUT_DIR = folder
    return mt.load_cell("main", 0).set_index("para_id")[mt.LABELS]

base = {m: baseline(f) for m, f in MODELS.items()}
common = sorted(set.intersection(*[set(b.dropna().index) for b in base.values()]))
base = {m: b.loc[common] for m, b in base.items()}
print(len(common), "paragraphs coded by all four models")

rows = []
for a, b in itertools.combinations(MODELS, 2):
    r = {"pair": f"{a} – {b}"}
    for l in mt.LABELS:
        r[l] = mt.alpha(base[a][l].to_numpy(float), base[b][l].to_numpy(float))
    r["foundations (mean)"] = np.mean([r[f] for f in mt.FOUNDATIONS])
    rows.append(r)
pairs = pd.DataFrame(rows).set_index("pair")

# human-human, same paragraphs
h = mt.load_humans()
h = h[h["para_id"].isin(common)]
hh = {}
for l in mt.LABELS:
    coders = [h[f"{mt.HUMAN_COL[l]}_{c}"].to_numpy(float) for c in mt.CODERS]
    hh[l] = np.mean([mt.alpha(x, y) for x, y in itertools.combinations(coders, 2)])
hh["foundations (mean)"] = np.mean([hh[f] for f in mt.FOUNDATIONS])

cross_model = pd.DataFrame({
    "LLM-LLM mean": pairs.mean(),
    "LLM-LLM min": pairs.min(),
    "LLM-LLM max": pairs.max(),
    "human-human mean": pd.Series(hh),
}).loc[LABELS].round(3)

pairs.round(3).to_csv(f"{OUT}/cross_model_pairs.csv")
cross_model.to_csv(f"{OUT}/cross_model_summary.csv")
display(pairs.round(2))
cross_model

## 2. Table 1 for each model (needed for Figures 3 and 4)

Saved as `table1_baseline.csv` in each model's folder. The ensemble row is the mean over the four 3-model ensembles.

In [ ]:
t1 = {}
for m, f in MODELS.items():
    mt.OUT_DIR = f
    t1[m] = mt.table1()
    t1[m].to_csv(f"{f}/table1_baseline.csv")

if os.path.exists(ENSEMBLE_TABLE1):
    t1[ENSEMBLE_NAME] = (pd.read_csv(ENSEMBLE_TABLE1)
                           .groupby("label", sort=False).mean(numeric_only=True))
else:
    print("no ensemble file found; run make_ensembles.py to add the ensemble")

pd.concat({m: t[["alpha_vs_majority", "alpha_vs_coders_mean", "llm_share_present"]]
           for m, t in t1.items()}, axis=1).round(2)

## 3. Figure 3: agreement with the human majority vote (baseline condition)

In [ ]:
mf.performance_models(t1, "fig3_majority_models", OUT)
from IPython.display import Image
Image(f"{OUT}/fig3_majority_models.png", width=850)

## 4. Figure 4: LLM–human vs. human–human agreement (baseline condition)

In [ ]:
mf.agreement_models(t1, "fig4_agreement_models", OUT)
Image(f"{OUT}/fig4_agreement_models.png", width=850)

## 5. Table 2: each condition vs. the baseline

First one full table per model (all labels; for the appendix), then the average over the four models with the range across models. The paired bootstrap takes a minute or two per model.

Columns: `alpha_with_baseline` = α between the condition's labels and the baseline labels (same model); `delta_vs_baseline` = change in α with the majority vote. If a model has a second baseline run (`main_0shot_validation_rep2.csv`), its row `main x 0-shot (run 2)` shows the run-to-run agreement.

In [ ]:
t2 = {}
for m, f in MODELS.items():
    mt.OUT_DIR = f
    t2[m] = mt.table2(labels=LABELS)
    t2[m].to_csv(f"{f}/table2_all_labels.csv")
    print("done:", m)

In [ ]:
stacked = pd.concat(t2, names=["model"]).reset_index()
COLS = ["alpha_with_baseline", "alpha_vs_majority", "delta_vs_baseline", "share_present"]
g = stacked.groupby(["condition", "label"], sort=False)[COLS]
t2_avg = pd.concat({"mean": g.mean(), "min": g.min(), "max": g.max()}, axis=1).round(3)
t2_avg.to_csv(f"{OUT}/table2_averaged.csv")
stacked.to_csv(f"{OUT}/table2_all_models.csv", index=False)

# how many models show a change whose 95% CI excludes zero, per condition and label
sig = (stacked.assign(sig=(stacked["delta_ci_low"] > 0) | (stacked["delta_ci_high"] < 0))
              .groupby(["condition", "label"], sort=False)["sig"].sum().unstack())
sig.to_csv(f"{OUT}/table2_n_models_significant.csv")

t2_avg.xs("mean", axis=1, level=0).unstack("label")[["alpha_with_baseline", "delta_vs_baseline"]] \
      .loc[:, (slice(None), ["moral_rhetoric", "foundations (mean)"])].round(2)

### LaTeX for the main-text Table 2

Moral rhetoric and the foundation mean. Each cell: mean over the four models, with the range across models in parentheses.

In [ ]:
PROMPT = {"main": "Standard prompt", "paraphrase": "Paraphrased", "reverse": "Reverse labels"}

def cell(cond, label, col, signed=False):
    m, lo, hi = (t2_avg.loc[(cond, label), (s, col)] for s in ("mean", "min", "max"))
    if pd.isna(m):
        return "--"
    f = (lambda x: f"{x:+.2f}".replace("-", "$-$")) if signed else (lambda x: f"{x:.2f}")
    return f"{f(m)} \\scriptsize({f(lo)}, {f(hi)})"

conds = [c for c in t2_avg.index.get_level_values(0).unique() if "baseline" not in c]
lines = []
prev = None
for c in conds:
    if "run 2" in c:
        prompt, shots = "Standard prompt", "0 (run 2)"
    else:
        v, k = c.split(" x ")
        prompt, shots = PROMPT[v], k.replace("-shot", "")
    if prev is not None and prompt != prev:
        lines.append("\\addlinespace")
    row = [prompt if prompt != prev else "", shots]
    for lab in ["moral_rhetoric", "foundations (mean)"]:
        row += [cell(c, lab, "alpha_with_baseline"), cell(c, lab, "delta_vs_baseline", signed=True)]
    lines.append(" & ".join(row) + " \\\\")
    prev = prompt

tex = r'''\begin{table}[htbp]
\centering
\caption{Sensitivity of LLM annotations to prompt variant and number of examples, averaged over four LLMs}
\label{tab:conditions}
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{llcccc}
\toprule
 & & \multicolumn{2}{c}{Moral rhetoric} & \multicolumn{2}{c}{Moral foundations (mean)} \\
\cmidrule(lr){3-4}\cmidrule(lr){5-6}
Prompt & Shots & $\alpha$ with baseline & $\Delta\alpha$ vs.\ majority & $\alpha$ with baseline & $\Delta\alpha$ vs.\ majority \\
\midrule
''' + "\n".join(lines) + r'''
\bottomrule
\end{tabular}
\par\vspace{4pt}
\begin{minipage}{0.95\linewidth}\footnotesize
\textit{Note:} Krippendorff's $\alpha$ (nominal), $N = 250$ paragraphs. Each cell shows the mean over four LLMs (gpt-6-luna, Claude Haiku 5.5, Gemma-4-31B-it, Qwen2.5-32B-Instruct), with the minimum and maximum across models in parentheses. ``$\alpha$ with baseline'' is the agreement between a condition's labels and the same model's labels in the baseline condition (standard prompt, zero-shot). ``$\Delta\alpha$ vs.\ majority'' is the change in agreement with the human majority vote relative to the baseline condition.
\end{minipage}
\end{table}
'''
open(f"{OUT}/table2_averaged.tex", "w").write(tex)
print(tex)

## 6. Sources of variation (summary for part 3)

Mean α between two sets of labels that differ in only one respect, averaged over models and labels in each group:

- **Run-to-run**: same model, same condition, run twice (only if `rep2` files exist).
- **Prompt variant**: same model, 0-shot, paraphrased or reverse-label prompt vs. the standard prompt.
- **Few-shot examples**: same model, standard prompt, 2- or 5-shot vs. 0-shot.
- **Model**: same condition (baseline), different models.
- **Human coders**: two human coders (reference).

In [ ]:
def within(conds):
    d = stacked[stacked["condition"].isin(conds)]
    return d.groupby("label", sort=False)["alpha_with_baseline"].mean()

sources = pd.DataFrame({
    "run-to-run": within(["main x 0-shot (run 2)"]),
    "prompt variant": within(["paraphrase x 0-shot", "reverse x 0-shot"]),
    "few-shot examples": within(["main x 2-shot", "main x 5-shot"]),
    "model": cross_model["LLM-LLM mean"],
    "human coders": cross_model["human-human mean"],
}).loc[LABELS].round(3)
sources.to_csv(f"{OUT}/sources_of_variation.csv")
sources